In [1]:
pip install trl

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# Adjust this path to the exact location of your file
JSONL_PATH = "/content/drive/MyDrive/UCL/SNLP/TED2025/multi_way.jsonl"

In [3]:
import json

print(f"Attempting to load data from: {JSONL_PATH}")

try:
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5: # Print only the first 5 lines
                break
            print(json.loads(line))
except FileNotFoundError:
    print(f"Error: The file at {JSONL_PATH} was not found. Please ensure the path is correct and your Google Drive is mounted.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

Attempting to load data from: /content/drive/MyDrive/UCL/SNLP/TED2025/multi_way.jsonl
{'talk_id': 'amber_mcreynolds_voting_doesn_t_need_to_be_so_hard_let_s_redesign_it', 'para_data': {'fr': 'Voter peut être difficile', 'en': 'Voting can be hard.', 'ru': 'Голосовать не всегда легко.', 'hu': 'Szavazni nem egyszerű.'}, 'Parallelism': 4, 'lang_list': ['fr', 'en', 'ru', 'hu'], 'timestamp': 8585}
{'talk_id': 'amber_mcreynolds_voting_doesn_t_need_to_be_so_hard_let_s_redesign_it', 'para_data': {'fr': 'Cela a été difficile,', 'en': "It's been hard -", 'ru': 'С начала развития демократии\nэто было затруднительно,', 'hu': 'Nehéz volt idáig –'}, 'Parallelism': 4, 'lang_list': ['fr', 'en', 'ru', 'hu'], 'timestamp': 10493}
{'talk_id': 'amber_mcreynolds_voting_doesn_t_need_to_be_so_hard_let_s_redesign_it', 'para_data': {'fr': 'parfois douloureux, parfois impossible,', 'en': 'sometimes painful, sometimes impossible -', 'hu': 'néha fájdalmas, néha lehetetlen –'}, 'Parallelism': 3, 'lang_list': ['fr', '

In [4]:
from collections import defaultdict
from datasets import IterableDataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer,SFTConfig
from peft import LoraConfig, get_peft_model

In [5]:

# -----------------
# Config
# -----------------

REQ_LANGS = ["en","es","zh"]

# Continued-pretrain settings
USE_TAGS = False          # <-- flip to False to try "no tags"
CHUNK_TOKENS = 1024     # long chunks for efficiency (try 2048/4096)
MAX_STEPS = 1000

OUTDIR = "/content/drive/MyDrive/UCL/SNLP" + ("-tags" if USE_TAGS else "-notags")


In [6]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=hf_token)

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "meta-llama/Llama-2-7b-hf"

# -----------------
# Tokenizer
# -----------------
tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# -----------------
# Model
# -----------------
rank =0

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map={"": rank}
)
# -----------------
# Lora
# -----------------
lora_config = LoraConfig(
    r=16, # Rank: Controls the capacity of the adapter
    lora_alpha=32, # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Target attention routing, freeze factual MLPs
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [7]:
# -----------------
# Helpers
# -----------------
def has_all_langs(obj, req_langs=REQ_LANGS):
    pd = obj.get("para_data", {})
    return all((l in pd) and (pd[l] is not None) and (str(pd[l]).strip() != "") for l in req_langs)

def normalize_pd(para_data):
    """Keep only non-empty strings."""
    out = {}
    for l, t in para_data.items():
        if t is None:
            continue
        s = str(t).strip()
        if s:
            out[l] = s
    return out

def format_segment(para_data, use_tags: bool, lang_order=None):
    """
    Turn one JSON row's para_data into text to be learned via next-token prediction.
    - use_tags=False: just values concatenated
    - use_tags=True: "<en>\\ntext" style tags (plain text, no tokenizer resize)
    """
    pd = normalize_pd(para_data)
    if not pd:
        return ""

    if lang_order is None:
        langs = list(pd.keys())
    else:
        langs = [l for l in lang_order if l in pd]

    if use_tags:
        parts = [f"<{l}>\n{pd[l]}" for l in langs]
    else:
        parts = [pd[l] for l in langs]

    # EOS boundary between subtitle “rows”
    return "\n\n".join(parts) + tok.eos_token

# -----------------
# Optional: stats on strict 7-way rows
# -----------------
rows = 0
talk_ids = set()
tok_counts_by_lang = defaultdict(int)

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if not has_all_langs(obj):
            continue

        rows += 1
        talk_ids.add(obj.get("talk_id"))

        pd = obj["para_data"]
        for l in REQ_LANGS:
            tok_counts_by_lang[l] += len(tok.encode(pd[l], add_special_tokens=False))

import torch.distributed as dist
is_dist = dist.is_available() and dist.is_initialized()
rank = dist.get_rank() if is_dist else 0

if rank == 0:
    # your stats counting + prints

	print("Strict rows kept (all REQ_LANGS present):", rows)
	print("Unique talks (strict):", len(talk_ids))
	print("Token counts by lang (strict):", dict(tok_counts_by_lang))
	print("Total tokens (strict, sum over langs):", sum(tok_counts_by_lang.values()))


Strict rows kept (all REQ_LANGS present): 20172
Unique talks (strict): 247
Token counts by lang (strict): {'en': 217589, 'es': 272301, 'zh': 471131}
Total tokens (strict, sum over langs): 961021


In [8]:
# -----------------
# Per-talk buffered packing generator (works even if a row has only 2-3 langs)
# -----------------
def talk_chunk_generator(
    jsonl_path: str,
    tokenizer,
    use_tags: bool,
    chunk_tokens: int,
    lang_order=None,
    require_all_langs: bool = False,
):
    import torch.distributed as dist

    is_dist = dist.is_available() and dist.is_initialized()
    rank = dist.get_rank() if is_dist else 0
    world = dist.get_world_size() if is_dist else 1

    buffer_text = ""
    buffer_tok = 0
    current_talk = None

    def flush():
        nonlocal buffer_text, buffer_tok
        if buffer_text:
            yield {"text": buffer_text}
        buffer_text, buffer_tok = "", 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            # shard by line number across ranks
            if idx % world != rank:
                continue

            if not line.strip():
                continue

            obj = json.loads(line)

            if require_all_langs and (not has_all_langs(obj)):
                continue

            talk_id = obj.get("talk_id")
            para_data = obj.get("para_data", {})

            seg = format_segment(para_data, use_tags=use_tags, lang_order=lang_order)
            if not seg:
                continue

            # cheap token estimate (avoid CPU tokenization here)
            seg_tok = len(seg) // 4

            # talk boundary -> flush to avoid mixing talks (within this rank's shard)
            if current_talk is None:
                current_talk = talk_id
            elif talk_id != current_talk:
                yield from flush()
                current_talk = talk_id

            if seg_tok >= chunk_tokens:
                yield from flush()
                yield {"text": seg}
                continue

            if buffer_tok + seg_tok > chunk_tokens:
                yield from flush()
                buffer_text = seg
                buffer_tok = seg_tok
            else:
                buffer_text += seg
                buffer_tok += seg_tok

    yield from flush()

# Choose whether you want to restrict training to rows where all 7 langs exist:
REQUIRE_ALL_7WAY = True  # recommended False for more data

train_ds = IterableDataset.from_generator(
    lambda: talk_chunk_generator(
        JSONL_PATH,
        tokenizer=tok,
        use_tags=USE_TAGS,
        chunk_tokens=CHUNK_TOKENS,
        lang_order=REQ_LANGS,          # fixed order; set None to keep dict order
        require_all_langs=REQUIRE_ALL_7WAY,
    )
)

In [9]:
# -----------------
# Lora training
# -----------------

sft_args = SFTConfig(
    max_length=CHUNK_TOKENS,
    packing=False,
    dataset_text_field="text",   # default is "text", but explicit is fine
    output_dir=OUTDIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    max_steps=MAX_STEPS,
    logging_steps=50,
    bf16=True, #set dependent on gpu
    fp16=False, #set opposite above
    optim="adamw_torch",
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    warmup_steps=int(0.03 * MAX_STEPS),
    report_to="none",
    gradient_checkpointing=False, #memory saving
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    processing_class=tok,
    peft_config=lora_config, # tokenizer/processor goes here in current API
)



In [10]:
#Print a couple of samples to verify formatting looks reasonable before training
it = iter(train_ds)
for i in range(2):
    ex = next(it)
    print("=== SAMPLE", i, "===")
    print(ex["text"][:1000])


=== SAMPLE 0 ===
I'd like you all

Me gustaría que todos

我想各位</s>to ask yourselves a question

se preguntaran

問自己一個你可能從來未問過嘅問題</s>What is possible with the human voice?

¿Cuál es la potencialidad de la voz?

人類嘅聲音有可以做到啲咩？</s>♪ Ooh baby ♪

♪ Ooh nena ♪

♪ 哦奀咕 ♪</s>♪ baby ♪

♪ nena ♪

♪ 奀咕 ♪</s>♪ baby ♪

♪ bebé ♪

♪ 奀咕 ♪</s>♪ baby ♪ (Baby crying)

♪ bebé ♪ 
(Bebé llorando)

♪ 奀咕 ♪（嬰兒哭聲）</s>♪ baby ♪ (Baby crying)

♪ bebé ♪ 
(Bebé llorando)

♪ 奀咕 ♪（嬰兒哭聲）</s>♪ baby ♪ (Cat meowing)

♪ bebé ♪ 
(Maullido)

♪ 奀咕 ♪（貓叫）</s>(Dog barking)

(Ladrido)

（狗吠）</s>Yeah.

¡Sí!

耶！</s>(Boomerang noises)

(Boomerang)
(Chirrido e impacto)

（迴旋鏢聲）</s>It was coming straight for me.
I had to. It was, yeah.

(Risas) Venía directo hacia mí.
Tenía que hacerlo. Así fue, sí.

頭先嗰啲全部都係嚟自我嘅聲音</s>As you can probably well imagine,

Como bien podrán imaginar,

所以你哋大概都可以想像到</s>I was a strange child.

yo era un niño raro.

我細個係一個好奇怪嘅細路</s>(Laughter)

(Risas)

（笑聲）</s>Because the thing is,
I was constantly trying

Lo cier

In [11]:
trainer.train()
trainer.save_model(f"{OUTDIR}/final")
tok.save_pretrained(f"{OUTDIR}/final")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
50,1.903060
100,1.346890
150,1.277443
200,1.229716
250,1.186492
300,1.145646
350,1.104020
400,1.059121
450,1.027352
500,0.996941


('/content/drive/MyDrive/UCL/SNLP-notags/final/tokenizer_config.json',
 '/content/drive/MyDrive/UCL/SNLP-notags/final/tokenizer.json')